In [1]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

In [2]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import mplfinance as mpf
import pandas_ta as ta
import pygame
import os
import pytz
import json
import re
import random
import pickle

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.callbacks import Callback, ModelCheckpoint
from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from tensorflow import keras
from tensorflow.keras import layers, optimizers, models
from scipy.signal import find_peaks

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 25  # 25 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 15  # 15 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [4]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [5]:
 session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [6]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [7]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [8]:
two_interval_minutes = 2

nf_index_symbol, nf_quantity = get_index_symbol_and_quantity("Nifty")
bnf_index_symbol, bnf_quantity = get_index_symbol_and_quantity("Bank Nifty")

#nf_train_df = fetch_candle_data(100, nf_index_symbol, two_interval_minutes)
#bnf_train_df = fetch_candle_data(100, bnf_index_symbol, two_interval_minutes)

nf_train_df = fetch_train_candle_data(5, nf_index_symbol, two_interval_minutes)
bnf_train_df = fetch_train_candle_data(5, bnf_index_symbol, two_interval_minutes)

print(len(nf_train_df), len(bnf_train_df))

nf_train_df = nf_train_df.drop_duplicates(subset='datetime', keep='first')
bnf_train_df = bnf_train_df.drop_duplicates(subset='datetime', keep='first')

print(len(nf_train_df), len(bnf_train_df))

63870 63870
63494 63494


In [9]:
class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame, base_interval_minutes: int):
        """
        df: DataFrame with columns [datetime, open, high, low, close, volume (optional)]
            'datetime' is a Unix timestamp in seconds.
        base_interval_minutes: The base timeframe in minutes (e.g. 2, 5, 15, etc.).
        """
        self.df = df.copy()
        self.base_interval = base_interval_minutes

        # We no longer need an htf_map since higher timeframe features are removed
        # self.htf_map = { ... }  # Removed

    def preprocess_datetime(self):
        """
        1) Convert Unix timestamp to local time (Asia/Kolkata).
        2) Check for duplicates/missing timestamps.
        3) Sort by time and set 'datetime' as the index.
        """
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        # Check for duplicates/missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        """
        Optionally drop 'volume' column if it contains zeros or NaNs.
        Adjust if you prefer different volume handling.
        """
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_base_timeframe_features(self):
        """
        Compute a few standard technical indicators on the base timeframe:
        RSI, MACD, Bollinger Bands, ATR, etc.
        """
        self.df.sort_index(inplace=True)

        # RSI on base timeframe close
        self.df['rsi_base'] = ta.rsi(self.df['close'], length=14)

        # MACD
        macd = ta.macd(self.df['close'], fast=12, slow=26, signal=9)
        self.df['macd'] = macd['MACD_12_26_9']
        self.df['macd_h'] = macd['MACDh_12_26_9']
        self.df['macd_s'] = macd['MACDs_12_26_9']

        # Bollinger Bands
        bbands = ta.bbands(self.df['close'], length=20, std=2)
        self.df['bb_lower'] = bbands['BBL_20_2.0']
        self.df['bb_mid'] = bbands['BBM_20_2.0']
        self.df['bb_upper'] = bbands['BBU_20_2.0']

        # ATR (for baseline volatility measure)
        self.df['atr_base'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        )

        return self

    def add_candlestick_patterns(self):
        """
        Detect a few common candlestick patterns.
        We store them as numeric columns (0 = not present, 1 = bullish pattern, -1 = bearish pattern, etc.)
        so that the pipeline remains consistent with numeric-only scaling.
        """
        self.df.sort_index(inplace=True)

        # Example: Doji detection (very simplistic)
        # We check if the open and close are nearly the same
        body_size = (self.df['close'] - self.df['open']).abs()
        candle_range = self.df['high'] - self.df['low']
        self.df['candle_doji'] = np.where(
            (body_size <= 0.1 * candle_range), 1.0, 0.0
        )

        # Example: Engulfing pattern (bullish and bearish)
        # - Bullish Engulfing: today's open < yesterday's close, today's close > yesterday's open
        # - Bearish Engulfing: today's open > yesterday's close, today's close < yesterday's open
        # We shift the open/close by 1 day to compare
        self.df['open_shift1'] = self.df['open'].shift(1)
        self.df['close_shift1'] = self.df['close'].shift(1)

        cond_bull_engulf = (self.df['open'] < self.df['close_shift1']) & (self.df['close'] > self.df['open_shift1'])
        cond_bear_engulf = (self.df['open'] > self.df['close_shift1']) & (self.df['close'] < self.df['open_shift1'])

        self.df['candle_engulf'] = np.where(cond_bull_engulf, 1.0,
                                    np.where(cond_bear_engulf, -1.0, 0.0))

        # Drop helper columns to keep dataset clean
        self.df.drop(['open_shift1', 'close_shift1'], axis=1, inplace=True)

        return self

    def add_swing_points(self, left_bars=3, right_bars=3):
        """
        Detect local swing highs and lows.
        left_bars/right_bars define how many bars on each side must be lower/higher (for highs/lows).
        """
        self.df.sort_index(inplace=True)

        # For each row, check if 'high' is greater than the preceding and following N bars
        rolling_high = self.df['high'].rolling(left_bars+1, center=False).max()
        rolling_low = self.df['low'].rolling(left_bars+1, center=False).min()

        # We'll do a simplistic approach: shift rolling computations accordingly
        # A more robust approach might be to iterate row by row, but we'll keep it vectorized for brevity.

        # Swing High
        # Compare current high to future bars as well, so we do a forward rolling
        forward_high = self.df['high'].shift(-right_bars).rolling(right_bars+1, center=False).max()
        self.df['is_swing_high'] = 0.0
        cond_high = (self.df['high'] == rolling_high) & (self.df['high'] == forward_high)
        self.df.loc[cond_high, 'is_swing_high'] = 1.0

        # Swing Low
        forward_low = self.df['low'].shift(-right_bars).rolling(right_bars+1, center=False).min()
        self.df['is_swing_low'] = 0.0
        cond_low = (self.df['low'] == rolling_low) & (self.df['low'] == forward_low)
        self.df.loc[cond_low, 'is_swing_low'] = 1.0

        return self

    def add_range_breakout_features(self, window=20):
        """
        Add Donchian channels, breakout flags, and a simple measure of range expansion.
        """
        self.df.sort_index(inplace=True)

        # Donchian Channels
        self.df['donchian_high'] = self.df['high'].rolling(window).max()
        self.df['donchian_low'] = self.df['low'].rolling(window).min()
        self.df['donchian_range'] = self.df['donchian_high'] - self.df['donchian_low']

        # Breakout flags
        #  1.0 if close > donchian_high, -1.0 if close < donchian_low, else 0.0
        self.df['donchian_breakout'] = np.where(
            self.df['close'] > self.df['donchian_high'], 1.0,
            np.where(self.df['close'] < self.df['donchian_low'], -1.0, 0.0)
        )

        # Range expansion/compression example
        # Rolling std of (high - low)
        self.df['range_expansion'] = (self.df['high'] - self.df['low']).rolling(window).std()

        return self

    def add_momentum_features(self):
        """
        Add additional momentum-based indicators like Stochastics, ADX, CCI, etc.
        (We already have RSI, MACD in add_base_timeframe_features, but you can unify them here if you prefer.)
        """
        self.df.sort_index(inplace=True)

        # Stochastic
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14, d=3
        )
        self.df['stoch_k'] = stoch['STOCHk_14_3_3']
        self.df['stoch_d'] = stoch['STOCHd_14_3_3']

        # ADX (Average Directional Movement Index)
        adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
        self.df['adx'] = adx_data['ADX_14']
        self.df['diplus'] = adx_data['DMP_14']
        self.df['diminus'] = adx_data['DMN_14']

        # CCI (Commodity Channel Index)
        self.df['cci'] = ta.cci(self.df['high'], self.df['low'], self.df['close'], length=20)

        return self

    def add_volatility_features(self, window=20):
        """
        Add additional volatility-based features, e.g. historical volatility, Z-score of returns, etc.
        """
        self.df.sort_index(inplace=True)

        # Historical volatility (using close-to-close log returns)
        self.df['log_return'] = np.log(self.df['close'] / self.df['close'].shift(1))
        self.df['hist_vol'] = self.df['log_return'].rolling(window).std() * np.sqrt(252)  # Annualized approximation

        # Z-Score of Price Changes over 'window'
        rolling_mean = self.df['log_return'].rolling(window).mean()
        rolling_std = self.df['log_return'].rolling(window).std()
        self.df['zscore_return'] = (self.df['log_return'] - rolling_mean) / (rolling_std + 1e-9)

        return self

    def add_volume_features(self):
        """
        Add volume-based features if 'volume' is present.
        """
        if 'volume' not in self.df.columns:
            return self  # No volume data, skip

        self.df.sort_index(inplace=True)

        # On-Balance Volume
        self.df['obv'] = ta.obv(self.df['close'], self.df['volume'])

        # Volume spike detection:
        # Compare current volume to rolling average
        vol_mean = self.df['volume'].rolling(20).mean()
        vol_std = self.df['volume'].rolling(20).std()
        self.df['vol_spike'] = np.where(
            (self.df['volume'] > (vol_mean + 2 * vol_std)), 1.0, 0.0
        )

        # VWAP over the day (if you have daily or session logic, you'd groupby date)
        # Here, for a simpler approach, we do a rolling 1-day approximation:
        # This might not be perfectly accurate if you have continuous 24h data, but it’s a start.
        # Usually you’d reset VWAP each day or each session.
        typical_price = (self.df['high'] + self.df['low'] + self.df['close']) / 3.0
        self.df['cum_tp_vol'] = (typical_price * self.df['volume']).cumsum()
        self.df['cum_vol'] = self.df['volume'].cumsum()
        self.df['vwap'] = self.df['cum_tp_vol'] / (self.df['cum_vol'] + 1e-9)

        # Drop helper columns
        self.df.drop(['cum_tp_vol', 'cum_vol'], axis=1, inplace=True)

        return self

    def add_regime_features(self):
        """
        Classify each bar as 'trending' or 'ranging' (0 or 1) based on ADX, or volatility-based thresholds, etc.
        Keep it numeric for scaling (e.g., 0.0 or 1.0).
        """
        self.df.sort_index(inplace=True)

        # If ADX above 25 => trending, else ranging
        # (25 is just a typical reference threshold; pick whichever fits your strategy)
        if 'adx' not in self.df.columns:
            # If ADX wasn't yet computed, do it quickly
            adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
            self.df['adx'] = adx_data['ADX_14']

        self.df['regime_trend'] = np.where(self.df['adx'] >= 25, 1.0, 0.0)

        # Alternatively, you could have multiple columns for different regime classifications
        return self

    def add_time_features(self):
        """
        Extract cyclical or numeric time features (hour of day, day of week).
        We ensure they are numeric (float).
        """
        self.df.sort_index(inplace=True)

        # Because 'datetime' is now our index, we can access it via self.df.index
        # Create columns for hour, day of week
        self.df['hour'] = self.df.index.hour.astype(float)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(float)

        # Cyclical encoding for hour
        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24.0)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24.0)

        # Cyclical encoding for day_of_week
        self.df['dow_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7.0)
        self.df['dow_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7.0)

        # You can drop raw hour/day_of_week if you prefer only cyclical features
        # self.df.drop(['hour', 'day_of_week'], axis=1, inplace=True)

        return self

    def add_adaptive_targets_and_stops(self):
        """
        Instead of fixed (2 * ATR) / (1 * ATR), adapt based on current volatility or regime.
        For demonstration, we’ll say:
            - If regime is trending, target = 3 * ATR, stop = 1.5 * ATR
            - If regime is ranging, target = 1.5 * ATR, stop = 1 * ATR
        This is just an example logic.
        """
        self.df.sort_index(inplace=True)

        if 'atr_base' not in self.df.columns:
            self.df['atr_base'] = ta.atr(self.df['high'], self.df['low'], self.df['close'], length=14)

        # If we haven't computed 'regime_trend', do so
        if 'regime_trend' not in self.df.columns:
            self.add_regime_features()

        # Example adaptation
        is_trend = (self.df['regime_trend'] == 1.0)
        self.df['Target'] = np.where(is_trend, 3.0 * self.df['atr_base'], 1.5 * self.df['atr_base'])
        self.df['StopLoss'] = np.where(is_trend, 1.5 * self.df['atr_base'], 1.0 * self.df['atr_base'])

        return self

    def run_pipeline(self):
        """
        Run all steps except higher timeframe computations (now removed).
        You can rearrange the order as desired.
        """
        (self.preprocess_datetime()
             .clean_data()
             .add_base_timeframe_features()
             .add_candlestick_patterns()
             .add_swing_points()
             .add_range_breakout_features()
             .add_momentum_features()
             .add_volatility_features()
             .add_volume_features()            # only applies if 'volume' exists
             .add_regime_features()
             .add_time_features()
             .add_adaptive_targets_and_stops()
        )
        return self

    def get_processed_df(self):
        """
        Retrieve the final DataFrame (including any signals if label_signals() was called).
        We'll also ensure that all columns except 'Signal' are float with 2 decimal places.
        """
        # Drop rows that have NaN from lookbacks
        self.df.dropna(axis=0, how='any', inplace=True)

        # Drop 'Entry Price' and 'Exit Price' if you do not want them in final features
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        # Ensure 'Signal' is int; everything else float with 2 decimals
        if 'Signal' in self.df.columns:
            sig_col = self.df['Signal'].astype(int)
            self.df.drop(columns=['Signal'], inplace=True)
        else:
            sig_col = None

        # Convert all remaining columns to float
        for col in self.df.columns:
            self.df[col] = self.df[col].astype(float)

        # Round to 2 decimals
        self.df = self.df.round(2)

        # Reinsert Signal column if it exists
        if sig_col is not None:
            self.df['Signal'] = sig_col

        return self.df

nf_train_pipeline = FullFeaturePipeline(nf_train_df, two_interval_minutes)
nf_train_pipeline.run_pipeline()

nf_processed_train_df = nf_train_pipeline.get_processed_df()

bnf_train_pipeline = FullFeaturePipeline(bnf_train_df, two_interval_minutes)
bnf_train_pipeline.run_pipeline()

bnf_processed_train_df = bnf_train_pipeline.get_processed_df()

In [10]:
nf_processed_train_df.columns

Index(['open', 'high', 'low', 'close', 'rsi_base', 'macd', 'macd_h', 'macd_s',
       'bb_lower', 'bb_mid', 'bb_upper', 'atr_base', 'candle_doji',
       'candle_engulf', 'is_swing_high', 'is_swing_low', 'donchian_high',
       'donchian_low', 'donchian_range', 'donchian_breakout',
       'range_expansion', 'stoch_k', 'stoch_d', 'adx', 'diplus', 'diminus',
       'cci', 'log_return', 'hist_vol', 'zscore_return', 'regime_trend',
       'hour', 'day_of_week', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
       'Target', 'StopLoss'],
      dtype='object')

In [11]:
nf_processed_train_df

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,zscore_return,regime_trend,hour,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,Target,StopLoss
datetime,,,,,,,,,,,,,,,,,,,,,
2023-06-08 10:21:00,18769.30,18773.80,18767.10,18771.20,68.81,6.19,1.85,4.34,18739.99,18754.64,...,0.43,0.0,10.0,3.0,0.50,-0.87,0.43,-0.9,10.89,7.26
2023-06-08 10:23:00,18771.90,18775.60,18771.20,18772.10,69.39,6.76,1.94,4.82,18741.13,18756.21,...,-0.13,0.0,10.0,3.0,0.50,-0.87,0.43,-0.9,10.56,7.04
2023-06-08 10:25:00,18772.80,18772.80,18764.10,18765.40,60.39,6.59,1.42,5.18,18741.33,18756.84,...,-1.53,0.0,10.0,3.0,0.50,-0.87,0.43,-0.9,10.75,7.17
2023-06-08 10:27:00,18764.60,18771.20,18764.60,18770.20,64.00,6.77,1.28,5.50,18741.60,18757.84,...,0.79,0.0,10.0,3.0,0.50,-0.87,0.43,-0.9,10.68,7.12
2023-06-08 10:29:00,18771.10,18771.20,18767.40,18768.20,61.49,6.68,0.95,5.73,18741.88,18758.58,...,-0.57,0.0,10.0,3.0,0.50,-0.87,0.43,-0.9,10.30,6.87
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-18 15:21:00,24843.65,24856.95,24843.40,24855.75,48.83,-5.78,-1.40,-4.38,24833.63,24858.61,...,1.88,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.9,18.50,12.33
2024-10-18 15:23:00,24857.10,24876.40,24855.65,24875.15,59.75,-3.46,0.74,-4.20,24833.28,24859.24,...,2.43,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.9,19.40,12.93
2024-10-18 15:25:00,24874.55,24876.35,24865.80,24866.40,54.14,-2.30,1.52,-3.82,24833.40,24859.54,...,-1.13,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.9,19.14,12.76


In [12]:
bnf_processed_train_df

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,zscore_return,regime_trend,hour,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,Target,StopLoss
datetime,,,,,,,,,,,,,,,,,,,,,
2023-06-08 10:21:00,44439.80,44454.10,44420.40,44442.90,73.41,45.90,-2.06,47.95,44305.00,44384.70,...,0.10,1.0,10.0,3.0,0.50,-0.87,0.43,-0.9,77.51,38.76
2023-06-08 10:23:00,44448.20,44448.60,44411.60,44424.80,65.74,43.84,-3.29,47.13,44322.52,44391.39,...,-1.28,1.0,10.0,3.0,0.50,-0.87,0.43,-0.9,80.11,40.06
2023-06-08 10:25:00,44420.60,44420.60,44401.20,44411.40,60.69,40.66,-5.17,45.84,44333.83,44395.76,...,-0.95,1.0,10.0,3.0,0.50,-0.87,0.43,-0.9,79.39,39.70
2023-06-08 10:27:00,44407.20,44427.70,44407.20,44421.10,62.91,38.48,-5.88,44.37,44345.77,44400.38,...,0.27,1.0,10.0,3.0,0.50,-0.87,0.43,-0.9,78.02,39.01
2023-06-08 10:29:00,44423.40,44423.40,44399.10,44403.10,56.53,34.90,-7.57,42.47,44355.87,44403.44,...,-1.09,1.0,10.0,3.0,0.50,-0.87,0.43,-0.9,77.63,38.81
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-18 15:21:00,52111.70,52134.40,52087.25,52129.95,56.14,-3.92,3.12,-7.04,52025.25,52099.85,...,0.79,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.9,61.93,41.29
2024-10-18 15:23:00,52133.55,52153.35,52119.25,52147.35,58.96,0.54,6.07,-5.52,52026.14,52099.37,...,0.67,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.9,61.16,40.78
2024-10-18 15:25:00,52141.95,52155.25,52117.70,52121.10,53.39,1.94,5.97,-4.03,52026.76,52100.58,...,-1.16,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.9,60.82,40.54


In [13]:
# -----------------------------------------------------------------------------------
# 1) CONFIGURATION CELL
# -----------------------------------------------------------------------------------
# Define all parameters, hyperparameters, data, etc. in a single config dict for easy control.

config = {
    "data_dict": None,                    # { "instrument_name": pd.DataFrame, ... } - user sets externally
    "brokerage_dict": None,               # Add brokerage specific for each instrument
    "instrument_selection": "random",     # "random", "sequential", or a specific instrument name

    "initial_capital_multiplier": 3.0,    # Env capital = multiplier * max(close)
    "flip_position": True,                # if True, Sell on Long will close & flip to Short in one step
    "track_unrealized_in_reward": True,   # add incremental reward for open positions
    "unrealized_reward_weight": 0.3,      # weight for unrealized PnL in each step's reward

    # Reward mode: "raw", "pct", "log", or "clip"
    "reward_mode": "pct",
    "pct_base_capital": "initial",        # "current" or "initial" for 'pct'/'log' modes
    "clip_max_abs_reward": 1e5,           # used if reward_mode="clip"
    "reward_scaling_factor": 1.0,         # scale the realized + unrealized portion of reward

    # Slippage & Sharpe config
    "slippage_pct": 0.0002,               # e.g. 0.02% slippage
    "sharpe_reward_weight": 0.1,          # how strongly Sharpe ratio is factored into step reward

    # We keep max_drawdown_percent only to end episodes, if desired.
    # (No longer used for reward penalty, since we replaced drawdown with Sharpe.)
    "max_drawdown_percent": 0.5,          # end episode if drawdown >= 50% (None to disable)
    "stop_on_end_of_data": True,          # end episode when we pass last row of the chosen instrument
    "strict_capital_check": True,         # disallow new positions if capital < (price + brokerage)
    "observation_window": 30,             # how many past bars to include in each observation

    "use_risk_adjusted_logging": True     # track step returns so we can compute approximate Sharpe
}

RL Environment

In [14]:
# -----------------------------------------------------------------------------------
# 2) ENVIRONMENT CELL (With Slippage + Sharpe-based reward)
# -----------------------------------------------------------------------------------

import gym
import numpy as np
import pandas as pd
import random

class SingleInstrumentTradingEnv(gym.Env):
    """
    A robust environment to train on a SINGLE instrument at a time, with detailed logging.
    - If multiple instruments are provided in config["data_dict"], the environment will
      select one instrument (via config["instrument_selection"]) at reset().
    - This environment focuses on capital-based trading with brokerage, slippage, and
      a Sharpe-based reward component (instead of a drawdown penalty).
    - End conditions:
        1) capital <= 0
        2) drawdown >= max_drawdown_percent (if set)
        3) end of data
    - Slippage is controlled by config["slippage_pct"].
    - The reward each step is:
         step_reward = [ (realized + unrealized PnL) * reward_scaling_factor ]
                       + [ sharpe_reward_weight * approx_sharpe_so_far ]
      where approx_sharpe_so_far = mean(step_returns) / std(step_returns)
      step_returns are stored each step if use_risk_adjusted_logging=True.
    - We also log "points_captured" on each close or flip.
    """

    def __init__(self, config):
        super(SingleInstrumentTradingEnv, self).__init__()

        # Store the config dict
        self.config = config

        # 1) Data-related
        self.data_dict = self.config["data_dict"]
        if not self.data_dict or len(self.data_dict) == 0:
            raise ValueError("config['data_dict'] must have at least one instrument DataFrame.")

        self.instrument_selection = self.config["instrument_selection"]

        self.current_instrument = None
        self.df = None
        self.max_steps = None

        # 2) Trading & Reward params
        self.initial_capital_multiplier = self.config["initial_capital_multiplier"]
        self.flip_position = self.config["flip_position"]
        self.track_unrealized_in_reward = self.config["track_unrealized_in_reward"]
        self.unrealized_reward_weight = self.config["unrealized_reward_weight"]

        self.reward_mode = self.config["reward_mode"]
        self.pct_base_capital = self.config["pct_base_capital"]
        self.clip_max_abs_reward = self.config["clip_max_abs_reward"]
        self.reward_scaling_factor = self.config["reward_scaling_factor"]

        # Slippage & Sharpe
        self.slippage_pct = self.config.get("slippage_pct", 0.0)
        self.sharpe_reward_weight = self.config.get("sharpe_reward_weight", 0.0)

        self.max_drawdown_percent = self.config["max_drawdown_percent"]
        self.stop_on_end_of_data = self.config["stop_on_end_of_data"]
        self.strict_capital_check = self.config["strict_capital_check"]
        self.observation_window = self.config["observation_window"]
        self.use_risk_adjusted_logging = self.config["use_risk_adjusted_logging"]

        # 3) Internal trackers
        self.capital = 0.0
        self.initial_capital = 0.0
        self.current_step = 0
        self.max_capital_seen = 0.0
        self.position = 0           # 0=flat, 1=long, -1=short
        self.entry_price = 0.0
        self.unrealized_pnl = 0.0
        self.wins = 0
        self.losses = 0
        self.hold_step_count = 0
        self.trade_durations = []

        # 4) Logs
        self.capital_log = []
        self.win_percentage_log = []
        self.position_type_log = []
        self.step_returns = []       # for approximate Sharpe
        self.max_drawdown_logged = []
        self.step_logs = []

        # 5) Action/Observation spaces
        self.action_space = gym.spaces.Discrete(3)  # (0=Hold, 1=Buy, 2=Sell)
        self.observation_space = None  # Will be defined after picking instrument in reset()

    def reset(self):
        """Reset the environment state, pick an instrument, set capital, logs, etc."""
        self.current_instrument = self._select_instrument()
        self.brokerage_per_side = self.config["brokerage_dict"][self.current_instrument]
        self.df = self.data_dict[self.current_instrument]
        self.max_steps = len(self.df)

        self.initial_capital = self._init_capital(self.df)
        self.capital = self.initial_capital
        self.max_capital_seen = self.capital

        self.current_step = 0
        self.position = 0
        self.entry_price = 0.0
        self.unrealized_pnl = 0.0
        self.wins = 0
        self.losses = 0
        self.hold_step_count = 0
        self.trade_durations.clear()

        # Clear logs
        self.capital_log.clear()
        self.win_percentage_log.clear()
        self.position_type_log.clear()
        self.step_returns.clear()
        self.max_drawdown_logged.clear()
        self.step_logs.clear()

        # Build observation space shape, now we know how many features & observation_window
        sample_obs = self._get_observation()
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=sample_obs.shape, dtype=np.float32
        )

        return sample_obs

    def step(self, action):
        """
        Main step logic:
          - Calculate current price and apply slippage for any fills
          - Update unrealized PnL
          - Perform the action (Buy, Sell, Hold)
          - Compute realized PnL if closing or flipping
          - Compute incremental reward from realized + unrealized PnL
          - Add approximate Sharpe ratio into the reward
          - Check done conditions
          - Log step data
        """
        done = False
        step_reward = 0.0

        # If we're out of data, we're done
        if self.current_step >= self.max_steps:
            return self._get_observation(), 0.0, True, {}

        row_idx = min(self.current_step, self.max_steps - 1)
        current_price = self.df['close'].iloc[row_idx]

        # 1) Update unrealized PnL based on current market price (no slippage for ongoing hold)
        if self.position == 1:
            self.unrealized_pnl = current_price - self.entry_price
        elif self.position == -1:
            self.unrealized_pnl = self.entry_price - current_price
        else:
            self.unrealized_pnl = 0.0

        # Track how many steps we've been in a position
        if self.position != 0:
            self.hold_step_count += 1
        else:
            self.hold_step_count = 0

        unreal_r = 0.0
        if self.track_unrealized_in_reward:
            unreal_r = self._transform_pnl_to_reward(self.unrealized_pnl)

        # We'll detect position close or flip to compute realized PnL
        realized_pnl = 0.0
        points_captured = 0.0

        # 2) Interpret action with slippage
        #    We'll define a small helper inside for fill_price with slippage.
        def buy_fill_price(ref_price):
            return ref_price * (1.0 + self.slippage_pct)

        def sell_fill_price(ref_price):
            return ref_price * (1.0 - self.slippage_pct)

        # If we're flat
        if self.position == 0:
            if action == 1:  # Buy
                potential_fill = buy_fill_price(current_price)
                if self._can_open_position(potential_fill):
                    self.position = 1
                    self.entry_price = potential_fill
                    self.capital -= self.brokerage_per_side
            elif action == 2:  # Sell
                potential_fill = sell_fill_price(current_price)
                if self._can_open_position(potential_fill):
                    self.position = -1
                    self.entry_price = potential_fill
                    self.capital -= self.brokerage_per_side

        # If we're long
        elif self.position == 1:
            if action == 2:  # either close or flip to short
                # closing long => fill at sell slippage
                exit_fill = sell_fill_price(current_price)
                realized_pnl = exit_fill - self.entry_price
                self.capital += realized_pnl
                self.capital -= self.brokerage_per_side

                points_captured = exit_fill - self.entry_price

                if realized_pnl > 0:
                    self.wins += 1
                else:
                    self.losses += 1

                self.trade_durations.append(self.hold_step_count)

                if self.flip_position:
                    # flip to short => new short fill at buy_fill_price? Actually for a short, we 'sell'.
                    flip_fill = sell_fill_price(current_price)
                    if self._can_open_position(flip_fill):
                        self.position = -1
                        self.entry_price = flip_fill
                        self.capital -= self.brokerage_per_side
                        self.unrealized_pnl = 0.0
                        self.hold_step_count = 1
                    else:
                        self.position = 0
                        self.entry_price = 0.0
                        self.unrealized_pnl = 0.0
                        self.hold_step_count = 0
                else:
                    self.position = 0
                    self.entry_price = 0.0
                    self.unrealized_pnl = 0.0
                    self.hold_step_count = 0

        # If we're short
        else:
            if action == 1:  # close short or flip to long
                exit_fill = buy_fill_price(current_price)
                realized_pnl = self.entry_price - exit_fill
                self.capital += realized_pnl
                self.capital -= self.brokerage_per_side

                points_captured = self.entry_price - exit_fill

                if realized_pnl > 0:
                    self.wins += 1
                else:
                    self.losses += 1

                self.trade_durations.append(self.hold_step_count)

                if self.flip_position:
                    flip_fill = buy_fill_price(current_price)
                    if self._can_open_position(flip_fill):
                        self.position = 1
                        self.entry_price = flip_fill
                        self.capital -= self.brokerage_per_side
                        self.unrealized_pnl = 0.0
                        self.hold_step_count = 1
                    else:
                        self.position = 0
                        self.entry_price = 0.0
                        self.unrealized_pnl = 0.0
                        self.hold_step_count = 0
                else:
                    self.position = 0
                    self.entry_price = 0.0
                    self.unrealized_pnl = 0.0
                    self.hold_step_count = 0

        # 3) Realized reward portion
        realized_r = self._transform_pnl_to_reward(realized_pnl)

        # 4) Update capital peak for drawdown (if we still want to end episodes, but not for reward)
        self.max_capital_seen = max(self.max_capital_seen, self.capital)
        current_drawdown = (self.max_capital_seen - self.capital) / max(self.max_capital_seen, 1e-9)

        # 5) Approx Sharpe so far
        #    step_returns updated below, so let's hold the last or compute after we update capital_log
        #    We'll do it after we log step return for *this* step.
        #    We'll store step_reward interim, then add Sharpe after we have it.

        # 6) Sum up step reward from realized + unrealized
        base_step_reward = (realized_r + unreal_r * self.unrealized_reward_weight) * self.reward_scaling_factor

        # 7) Check end conditions
        if self.capital <= 0:
            done = True
        if (self.max_drawdown_percent is not None) and (current_drawdown >= self.max_drawdown_percent):
            done = True
        if self.stop_on_end_of_data and (self.current_step + 1 >= self.max_steps):
            done = True

        # 8) Logging
        self.capital_log.append(self.capital)
        total_trades = self.wins + self.losses
        if total_trades > 0:
            win_percent = (self.wins / total_trades) * 100.0
        else:
            win_percent = 0.0
        self.win_percentage_log.append(win_percent)

        if self.position == 1:
            pos_str = f"Long-{self.current_instrument}"
        elif self.position == -1:
            pos_str = f"Short-{self.current_instrument}"
        else:
            pos_str = "Flat"
        self.position_type_log.append(pos_str)
        self.max_drawdown_logged.append(current_drawdown)

        # track step returns for approximate Sharpe
        if self.use_risk_adjusted_logging:
            if len(self.capital_log) > 1:
                prev_cap = self.capital_log[-2]
                step_ret = (self.capital - prev_cap) / max(abs(prev_cap), 1e-9)
            else:
                step_ret = 0.0
            self.step_returns.append(step_ret)

        # Now compute approximate Sharpe ratio after including this step's return
        sharpe_approx = 0.0
        if self.use_risk_adjusted_logging and len(self.step_returns) > 1:
            rets = np.array(self.step_returns)
            avg_r = rets.mean()
            std_r = rets.std() + 1e-9
            sharpe_approx = avg_r / std_r

        # Final step reward
        step_reward = base_step_reward + self.sharpe_reward_weight * sharpe_approx

        # store step logs
        self.step_logs.append({
            "step": self.current_step,
            "capital": self.capital,
            "win_percent": win_percent,
            "position": pos_str,
            "unrealized_pnl": self.unrealized_pnl,
            "realized_pnl": realized_pnl,
            "points_captured": points_captured,
            "drawdown_fraction": current_drawdown,
            "approx_sharpe": sharpe_approx,
            "reward": step_reward
        })

        # 9) Advance and build next observation
        self.current_step += 1
        obs = self._get_observation()

        return obs, step_reward, done, {}

    def render(self, mode='human'):
        """Optional debug print."""
        if len(self.capital_log) > 0:
            idx = len(self.capital_log) - 1
            cap_str = f"{self.capital_log[idx]:.2f}"
            winp_str = f"{self.win_percentage_log[idx]:.2f}"
            pos_str = self.position_type_log[idx]
            dd_str = f"{self.max_drawdown_logged[idx]*100:.2f}%"
            print(f"Step: {self.current_step}, Cap: {cap_str}, Win%: {winp_str}, Pos: {pos_str}, DD: {dd_str}")
        else:
            print(f"Step: {self.current_step}, Cap: {self.capital:.2f}, Pos: [No log yet]")

    def close(self):
        pass

    # -------------------------------------------------------------------
    # EVALUATION / LOG METHODS
    # -------------------------------------------------------------------
    def get_evaluation_report(self):
        """
        Returns a dictionary with final (or current) evaluation metrics,
        including approximate Sharpe ratio, final capital, etc.
        """
        final_capital = self.capital_log[-1] if self.capital_log else self.capital
        net_profit = final_capital - self.initial_capital
        final_winrate = self.win_percentage_log[-1] if self.win_percentage_log else 0.0
        dd_list = self.max_drawdown_logged
        max_dd = max(dd_list) if len(dd_list) > 0 else 0.0

        report = {
            "instrument": self.current_instrument,
            "initial_capital": self.initial_capital,
            "final_capital": final_capital,
            "net_profit": net_profit,
            "final_winrate_percent": final_winrate,
            "capital_log": self.capital_log.copy(),
            "winrate_each_step": self.win_percentage_log.copy(),
            "position_each_step": self.position_type_log.copy(),
            "drawdown_each_step": dd_list.copy(),
            "max_drawdown_fraction": max_dd,
            "step_logs": self.step_logs.copy()
        }

        # approximate Sharpe ratio for entire episode
        if self.use_risk_adjusted_logging and len(self.step_returns) > 1:
            rets = np.array(self.step_returns)
            avg_r = rets.mean()
            std_r = rets.std()
            sharpe_approx = 0.0 if std_r < 1e-9 else (avg_r / std_r)
            report["approx_sharpe"] = sharpe_approx

        return report

    def get_step_logs_df(self):
        """DataFrame of step_logs for analysis."""
        return pd.DataFrame(self.step_logs)

    # -------------------------------------------------------------------
    # INTERNAL HELPERS
    # -------------------------------------------------------------------
    def _select_instrument(self):
        assets = list(self.data_dict.keys())
        if self.instrument_selection == "random":
            chosen = random.choice(assets)
        elif self.instrument_selection == "sequential":
            if not hasattr(self, '_seq_index'):
                self._seq_index = 0
            chosen = assets[self._seq_index % len(assets)]
            self._seq_index += 1
        else:
            if self.instrument_selection not in assets:
                raise ValueError(f"instrument_selection='{self.instrument_selection}' not in data_dict.")
            chosen = self.instrument_selection
        return chosen

    def _init_capital(self, df):
        max_close = df['close'].max()
        init_cap = self.initial_capital_multiplier * max_close
        return init_cap

    def _can_open_position(self, fill_price):
        """Check capital sufficiency if strict_capital_check=True."""
        if not self.strict_capital_check:
            return True
        cost_to_open = fill_price + self.brokerage_per_side
        return (self.capital >= cost_to_open)

    def _transform_pnl_to_reward(self, pnl):
        """Convert raw PnL => final reward (raw/pct/log/clip)."""
        mode = self.reward_mode

        if mode == "raw":
            return pnl

        elif mode == "pct":
            if self.pct_base_capital == "current":
                base = max(self.capital, 1e-9)
            else:
                base = max(self.initial_capital, 1e-9)
            return pnl / base

        elif mode == "log":
            if self.pct_base_capital == "current":
                base = max(self.capital, 1e-9)
            else:
                base = max(self.initial_capital, 1e-9)
            ret = pnl / base
            ret_clamped = max(ret, -0.999999)  # avoid log(<=0)
            return np.log1p(ret_clamped)

        elif mode == "clip":
            clipped_val = np.clip(pnl, -self.clip_max_abs_reward, self.clip_max_abs_reward)
            return (clipped_val / self.clip_max_abs_reward) * 100.0

        return pnl  # fallback

    def _get_observation(self):
        """
        Gather an observation by stacking the last `observation_window` rows
        plus capital, position, and unrealized PnL.
        """
        obs_frames = []
        for offset in range(self.observation_window):
            step_idx = self.current_step - (self.observation_window - 1 - offset)
            step_idx = max(step_idx, 0)
            step_idx = min(step_idx, self.max_steps - 1)
            row_data = self.df.iloc[step_idx].values  # e.g. [open, high, low, close, ...]
            obs_frames.append(row_data)

        # Flatten stacked window
        obs_data = np.concatenate(obs_frames, axis=0).tolist()

        # Append capital
        obs_data.append(self.capital)
        # Append position & unrealized
        obs_data.append(float(self.position))
        obs_data.append(self.unrealized_pnl)

        return np.array(obs_data, dtype=np.float32)

In [15]:
# Start with low volatility instruments and gradually move to volatile instruments to allow the model to generalize better.
config["data_dict"] = {
    "Nifty": nf_processed_train_df,
    "BankNifty": bnf_processed_train_df
    # add more instruments if needed
}

config["brokerage_dict"] = {
    "Nifty": 20.0,      # ₹20 per trade
    "BankNifty": 20.0,  # ₹20 per trade
}

config["instrument_selection"] = "sequential"  # "random" or "sequential" or "particular data name"

env = SingleInstrumentTradingEnv(config)
obs = env.reset()

done = False
while not done:
    action = env.action_space.sample()  # or from your RL policy
    obs, reward, done, info = env.step(action)
    env.render()


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Step: 1, Cap: 78800.35, Win%: 0.00, Pos: Short-Nifty, DD: 0.03%
Step: 2, Cap: 78751.94, Win%: 0.00, Pos: Long-Nifty, DD: 0.09%
Step: 3, Cap: 78751.94, Win%: 0.00, Pos: Long-Nifty, DD: 0.09%
Step: 4, Cap: 78751.94, Win%: 0.00, Pos: Long-Nifty, DD: 0.09%
Step: 5, Cap: 78751.94, Win%: 0.00, Pos: Long-Nifty, DD: 0.09%
Step: 6, Cap: 78751.94, Win%: 0.00, Pos: Long-Nifty, DD: 0.09%
Step: 7, Cap: 78696.03, Win%: 0.00, Pos: Short-Nifty, DD: 0.16%
Step: 8, Cap: 78646.93, Win%: 0.00, Pos: Long-Nifty, DD: 0.22%
Step: 9, Cap: 78646.93, Win%: 0.00, Pos: Long-Nifty, DD: 0.22%
Step: 10, Cap: 78646.93, Win%: 0.00, Pos: Long-Nifty, DD: 0.22%
Step: 11, Cap: 78646.93, Win%: 0.00, Pos: Long-Nifty, DD: 0.22%
Step: 12, Cap: 78646.93, Win%: 0.00, Pos: Long-Nifty, DD: 0.22%
Step: 13, Cap: 78646.93, Win%: 0.00, Pos: Long-Nifty, DD: 0.22%
Step: 14, Cap: 78646.93, Win%: 0.00, Pos: Long-Nifty, DD: 0.22%
Step: 15, Cap: 78609.12, Win%: 25.00, Pos: Short-Nifty, DD: 0.27%
Step: 16, Cap: 78609.12, Win%: 25.00, Pos: Sh

In [16]:
# After the episode, you can compute stats like approximate Sharpe ratio
if config["use_risk_adjusted_logging"]:
    rets = np.array(env.step_returns)
    avg_r = rets.mean()
    std_r = rets.std()
    sharpe_approx = avg_r / std_r if std_r > 1e-9 else 0
    print("Approx Sharpe:", sharpe_approx)

Approx Sharpe: -0.6443617290451245


In [17]:
report = env.get_evaluation_report()

print("Instrument:", report["instrument"])
print("Initial Capital:", report["initial_capital"])
print("Final Capital:", report["final_capital"])
print("Net Profit:", report["net_profit"])
print("Final Win Rate (%):", report["final_winrate_percent"])
print("Max Drawdown (%):", report["max_drawdown_fraction"] * 100.0)

Instrument: Nifty
Initial Capital: 78820.35
Final Capital: 39370.62716000056
Net Profit: -39449.72283999945
Final Win Rate (%): 14.698795180722893
Max Drawdown (%): 50.05017465667109


In [18]:
step_df = env.get_step_logs_df()
step_df

,step,capital,win_percent,position,unrealized_pnl,realized_pnl,points_captured,drawdown_fraction,approx_sharpe,reward
0,0,78800.35000,0.000000,Short-Nifty,0.00000,0.00000,0.00000,0.000254,0.000000,0.000000
1,1,78751.94134,0.000000,Long-Nifty,0.00000,-8.40866,-8.40866,0.000868,-0.999997,-0.100124
2,2,78751.94134,0.000000,Long-Nifty,-10.45442,0.00000,0.00000,0.000868,-0.707104,-0.070750
3,3,78751.94134,0.000000,Long-Nifty,-5.65442,0.00000,0.00000,0.000868,-0.577348,-0.057756
4,4,78751.94134,0.000000,Long-Nifty,-7.65442,0.00000,0.00000,0.000868,-0.499998,-0.050029
...,...,...,...,...,...,...,...,...,...,...
2573,2573,39426.43970,14.716526,Long-Nifty,-2.45710,0.00000,0.00000,0.499794,-0.644644,-0.064474
2574,2574,39426.43970,14.716526,Long-Nifty,-9.75710,0.00000,0.00000,0.499794,-0.644467,-0.064484
2575,2575,39426.43970,14.716526,Long-Nifty,-17.05710,0.00000,0.00000,0.499794,-0.644290,-0.064494
2576,2576,39426.43970,14.716526,Long-Nifty,-9.65710,0.00000,0.00000,0.499794,-0.644113,-0.064448


MAML Agent

In [19]:
# ---------------------------
# 3) ADVANCED ARCHITECTURE
# ---------------------------

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils import weight_norm
import higher
from einops import rearrange

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.conv1 = weight_norm(nn.Conv1d(in_channels, out_channels, kernel_size,
                                          padding=(kernel_size-1)*dilation, dilation=dilation))
        self.activation = nn.Mish()
        self.dropout = nn.Dropout(0.1)
        self.conv2 = weight_norm(nn.Conv1d(out_channels, out_channels, kernel_size,
                                          padding=(kernel_size-1)*dilation, dilation=dilation))
        self.res = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.activation(out)
        out = self.dropout(out)
        out = self.conv2(out)
        if self.res:
            residual = self.res(residual)
        return self.activation(out + residual[:, :, :-self.conv1.padding[0]])

class TCNStack(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=3):
        super().__init__()
        layers = []
        for i in range(num_layers):
            dilation = 2 ** i
            in_channels = input_size if i == 0 else hidden_size
            layers += [TCNBlock(in_channels, hidden_size, kernel_size=3, dilation=dilation)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class MarketAwareTransformer(nn.Module):
    def __init__(self, feature_count, obs_window, tcn_hidden=64, num_heads=8):
        super().__init__()
        self.obs_window = obs_window
        self.feature_count = feature_count

        # Temporal pattern extractor
        self.tcn = TCNStack(feature_count, tcn_hidden)

        # Transformer components
        self.pos_encoder = nn.Parameter(torch.zeros(1, obs_window, tcn_hidden))
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=tcn_hidden,
                nhead=num_heads,
                dim_feedforward=256,
                dropout=0.1,
                activation='gelu',
                batch_first=True
            ),
            num_layers=4
        )

        # Multi-scale trend detectors
        self.trend_conv3 = nn.Conv1d(tcn_hidden, tcn_hidden//2, kernel_size=3, padding=1)
        self.trend_conv7 = nn.Conv1d(tcn_hidden, tcn_hidden//2, kernel_size=7, padding=3)
        self.trend_merge = nn.Linear(tcn_hidden*2, tcn_hidden)

        # Position-aware attention
        self.position_attention = nn.MultiheadAttention(tcn_hidden, 4, batch_first=True)

        # State fusion
        self.state_net = nn.Sequential(
            nn.Linear(tcn_hidden + 3, 128),  # 3 for capital/position
            nn.LayerNorm(128),
            nn.Mish(),
            nn.Linear(128, 64)
        )

        # Policy head
        self.policy_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.Mish(),
            nn.Linear(32, 3)
        )

    def forward(self, x):
        batch_size = x.size(0)
        seq_part = x[:, :self.obs_window*self.feature_count].view(-1, self.obs_window, self.feature_count)
        state_part = x[:, -3:]  # Capital, position, unrealized

        # Temporal pattern extraction
        seq_t = rearrange(seq_part, 'b t f -> b f t')  # [Batch, Features, Time]
        tcn_out = self.tcn(seq_t)  # [Batch, Hidden, Time]
        tcn_out = rearrange(tcn_out, 'b h t -> b t h')  # [Batch, Time, Hidden]

        # Multi-scale trend detection
        trend3 = self.trend_conv3(tcn_out)
        trend7 = self.trend_conv7(tcn_out)
        trend_features = torch.cat([trend3, trend7], dim=-1)
        trend_features = self.trend_merge(trend_features)

        # Position-aware attention
        pos_attn, _ = self.position_attention(trend_features, trend_features, trend_features)

        # Transformer processing
        transformer_in = pos_attn + self.pos_encoder
        transformer_out = self.transformer(transformer_in)
        time_features = transformer_out.mean(dim=1)

        # State fusion
        fused = torch.cat([time_features, state_part], dim=1)
        state_embed = self.state_net(fused)

        return self.policy_head(state_embed)

# ---------------------------
# 4) ENHANCED META-LEARNER
# ---------------------------

class MarketMAMLTrainer:
    def __init__(self, config, feature_count):
        self.config = config
        self.feature_count = feature_count

        # Initialize model with market-aware architecture
        self.policy = MarketAwareTransformer(
            feature_count=feature_count,
            obs_window=config['observation_window']
        ).to(device)

        # Multi-task optimization
        self.meta_optimizer = optim.AdamW(self.policy.parameters(), lr=1e-3)
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.meta_optimizer, max_lr=1e-3, steps_per_epoch=100, epochs=100
        )

        # Market pattern memory
        self.brokerage_dict = config['brokerage_dict']
        self.instrument_features = {}  # Stores learned market embeddings

        # Normalization
        self.feature_scalers = None
        self.state_scalers = None

    def _prepare_env(self, instrument_name):
        """Create environment with instrument-specific brokerage"""
        env_config = self.config.copy()
        env_config['brokerage_dict'] = {instrument_name: self.brokerage_dict[instrument_name]}
        return SingleInstrumentTradingEnv(env_config)

    def _market_embedding(self, instrument_name):
        """Learn instrument-specific market embeddings"""
        if instrument_name not in self.instrument_features:
            # Initialize with hash-based features
            hash_val = hash(instrument_name) % 1000
            self.instrument_features[instrument_name] = nn.Parameter(
                torch.randn(1, 16, device=device) * (hash_val/1000)
            )
        return self.instrument_features[instrument_name]

    def _normalize_obs(self, obs):
        """Market-aware normalization with instrument-specific adjustments"""
        # Apply base normalization
        obs_norm = (obs - self.feature_scalers['mean']) / (self.feature_scalers['std'] + 1e-8)
        # Add instrument-specific scaling
        return obs_norm * self.market_scaling_factors[instrument_name]

    def meta_train(self, instruments_data, num_epochs=100):
        """Enhanced MAML training with market pattern adaptation"""
        # Initialize normalization
        self._init_normalization(instruments_data)

        for epoch in range(num_epochs):
            meta_loss = 0.0
            adapt_losses = []

            # Sample batch of markets
            instruments = random.sample(list(instruments_data.keys()), 4)

            with higher.innerloop_ctx(
                self.policy, self.meta_optimizer, copy_initial_weights=False
            ) as (fnet, diffopt):

                # Market adaptation loop
                for instrument in instruments:
                    env = self._prepare_env(instrument)
                    df = instruments_data[instrument]
                    market_emb = self._market_embedding(instrument)

                    # Fast adaptation
                    for adapt_step in range(3):
                        # Collect adaptation trajectories
                        obs = env.reset()
                        done = False
                        while not done:
                            obs_t = self._process_observation(obs, market_emb)
                            action = self._sample_action(fnet, obs_t)
                            next_obs, reward, done, _ = env.step(action)
                            # Update model with market-specific dynamics
                            loss = self._compute_adapt_loss(fnet, obs_t, action, reward)
                            diffopt.step(loss)
                            obs = next_obs

                    # Compute meta-loss on validation trajectory
                    val_loss, val_reward = self._validate_on_market(fnet, env, df, market_emb)
                    meta_loss += val_loss
                    adapt_losses.append(val_reward)

                # Meta-optimization step
                meta_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
                self.meta_optimizer.step()
                self.scheduler.step()
                self.meta_optimizer.zero_grad()

            # Dynamic market embedding update
            self._update_market_embeddings(instruments_data)

            print(f"Epoch {epoch+1} | Meta Loss: {meta_loss.item():.2f} | Avg Reward: {np.mean(adapt_losses):.2f}")

    def _process_observation(self, obs, market_emb):
        """Enhance observation with market embeddings"""
        obs_t = torch.FloatTensor(obs).to(device)
        seq_part = obs_t[:self.config['observation_window']*self.feature_count]
        state_part = obs_t[-3:]

        # Expand market embedding to match sequence length
        market_features = market_emb.repeat(seq_part.size(0)//16, 1)
        enhanced_seq = torch.cat([seq_part, market_features], dim=-1)

        return torch.cat([enhanced_seq, state_part])

    def _compute_adapt_loss(self, model, obs, action, reward):
        """Market-aware loss calculation"""
        logits = model(obs.unsqueeze(0))
        policy_loss = nn.CrossEntropyLoss()(logits, torch.LongTensor([action]).to(device))
        value_loss = nn.MSELoss()(logits.mean(), torch.FloatTensor([reward]).to(device))
        return policy_loss + 0.5 * value_loss

    def _validate_on_market(self, model, env, df, market_emb):
        """Market-specific validation with pattern scoring"""
        obs = env.reset()
        total_reward = 0
        pattern_scores = []

        while True:
            obs_t = self._process_observation(obs, market_emb)
            with torch.no_grad():
                logits = model(obs_t.unsqueeze(0))
                action = torch.argmax(logits, dim=-1).item()

            next_obs, reward, done, _ = env.step(action)

            # Pattern recognition scoring
            current_features = self._extract_market_patterns(obs)
            pattern_score = self._score_market_patterns(current_features)
            pattern_scores.append(pattern_score)

            total_reward += reward * (1 + pattern_score)  # Reward amplification

            if done:
                break
            obs = next_obs

        return -total_reward, total_reward  # Negative for loss minimization

    def _extract_market_patterns(self, obs):
        """Feature engineering for real-time pattern detection"""
        features = {
            'momentum': obs[..., [4, 11, 23, 26]].mean(),      # RSI, ATR, ADX, Z-score
            'volatility': obs[..., [10, 18, 25]].std(),       # BB width, Donchian, hist_vol
            'trend_strength': obs[..., [28, 29]].abs().mean() # Regime trend, DI+/-
        }
        return features

    def _score_market_patterns(self, features):
        """Score current market state based on learned patterns"""
        # Implement dynamic pattern scoring based on current regime
        trend_score = torch.sigmoid(features['trend_strength'] * 2 - 1)
        vol_score = torch.exp(-features['volatility'] * 0.5)
        return (trend_score * vol_score).item()

# ---------------------------
# 5) REAL-TIME INTEGRATION
# ---------------------------

class LiveMarketAdapter:
    def __init__(self, trained_model, config):
        self.config = config
        self.model = trained_model
        self.current_market_emb = None

        # Live market buffers
        self.price_buffer = []
        self.feature_buffer = []
        self.pattern_scores = []

    def process_live_data(self, new_df):
        """Process real-time data with market pattern detection"""
        # Maintain observation window
        self.feature_buffer = pd.concat([self.feature_buffer, new_df])[-self.config['observation_window']:]

        # Extract features
        features = new_df[self.config['feature_columns']].values
        self.price_buffer = new_df[['open', 'high', 'low', 'close']].values

        # Detect patterns
        current_patterns = self._detect_live_patterns()
        self.pattern_scores.append(current_patterns)

        return self._create_observation(features[-1], current_patterns)

    def _detect_live_patterns(self):
        """Real-time technical pattern detection"""
        prices = self.price_buffer[-30:]  # Last 30 periods for pattern detection

        # Example pattern: Detect breakout from consolidation
        recent_range = np.ptp(prices[-5:])  # Recent 5-period range
        prev_range = np.ptp(prices[-10:-5])
        consolidation_score = (prev_range - recent_range) / (prev_range + 1e-8)

        # Trend strength
        ma_short = np.mean(prices[-5:])
        ma_long = np.mean(prices[-20:])
        trend_score = (ma_short - ma_long) / (ma_long + 1e-8)

        return {
            'consolidation': np.tanh(consolidation_score * 3),
            'trend_strength': np.tanh(trend_score * 2),
            'volatility': np.std(prices[-10:]) / np.mean(prices[-10:])
        }

    def _create_observation(self, features, patterns):
        """Enhance observation with live patterns"""
        pattern_features = np.array([
            patterns['consolidation'],
            patterns['trend_strength'],
            patterns['volatility']
        ])
        return np.concatenate([features, pattern_features])

    def get_trading_signal(self, processed_obs):
        """Generate trading decision with market context"""
        with torch.no_grad():
            obs_t = torch.FloatTensor(processed_obs).to(device)
            market_emb = self._adapt_to_current_market()
            full_obs = self.model._process_observation(obs_t, market_emb)
            logits = self.model.policy(full_obs.unsqueeze(0))
            return torch.softmax(logits, dim=-1).cpu().numpy()

    def _adapt_to_current_market(self):
        """Online adaptation of market embeddings"""
        # Implement few-shot adaptation using recent patterns
        if len(self.pattern_scores) > 10:
            recent_patterns = torch.FloatTensor(self.pattern_scores[-10:]).mean(0)
            self.current_market_emb += recent_patterns * 0.1  # Dynamic adjustment
        return self.current_market_emb

# ---------------------------
# 6) TRAINING EXECUTION
# ---------------------------

# Initialize with proper market features
feature_columns = [
    'open', 'high', 'low', 'close', 'rsi_base', 'macd', 'macd_h', 'macd_s',
    'bb_lower', 'bb_mid', 'bb_upper', 'atr_base', 'candle_doji', 'candle_engulf',
    'is_swing_high', 'is_swing_low', 'donchian_high', 'donchian_low', 'donchian_range',
    'donchian_breakout', 'range_expansion', 'stoch_k', 'stoch_d', 'adx', 'diplus',
    'diminus', 'cci', 'log_return', 'hist_vol', 'zscore_return', 'regime_trend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos'
]

config.update({
    "feature_columns": feature_columns,
    "observation_window": 30,
    "brokerage_dict": {"Nifty": 20.0, "BankNifty": 20.0}
})

# Initialize trainer
market_trainer = MarketMAMLTrainer(
    config=config,
    feature_count=len(feature_columns) + 3  # Added pattern features
)

# Prepare datasets
train_datasets = {
    "Nifty": nf_processed_train_df,
    "BankNifty": bnf_processed_train_df
}

# Execute training
market_trainer.meta_train(train_datasets, num_epochs=100)

# ---------------------------
# 7) LIVE TRADING USAGE
# ---------------------------

class LiveTrader:
    def __init__(self, trained_model, config):
        self.config = config
        self.adapter = LiveMarketAdapter(trained_model, config)
        self.env = SingleInstrumentTradingEnv(config)

    def execute_live_trade(self, live_df, instrument_name):
        """Real-time trading execution with market adaptation"""
        # Configure environment for current instrument
        self.env.data_dict = {instrument_name: live_df}
        self.env.brokerage_dict = {instrument_name: self.config['brokerage_dict'][instrument_name]}

        # Initialize trading session
        obs = self.env.reset()
        trade_log = []

        while True:
            # Process market data and detect patterns
            processed_obs = self.adapter.process_live_data(live_df)

            # Get trading signal with current market context
            signal_probs = self.adapter.get_trading_signal(processed_obs)

            # Execute trade with risk management
            action = self._select_action_with_risk(signal_probs)
            next_obs, reward, done, _ = self.env.step(action)

            # Log trade execution
            trade_log.append({
                'timestamp': live_df.iloc[self.env.current_step]['timestamp'],
                'action': action,
                'confidence': signal_probs[action],
                'capital': self.env.capital,
                'position': self.env.position,
                'patterns': self.adapter.pattern_scores[-1]
            })

            if done:
                break
            obs = next_obs

        return trade_log

    def _select_action_with_risk(self, probs):
        """Risk-aware action selection with volatility scaling"""
        # Get volatility from recent patterns
        recent_vol = np.mean([p['volatility'] for p in self.adapter.pattern_scores[-5:]])

        # Adjust position sizing based on volatility
        position_scale = np.clip(0.5 / (recent_vol + 0.1), 0.1, 2.0)

        # Modify probabilities based on risk
        adjusted_probs = probs ** position_scale
        adjusted_probs /= adjusted_probs.sum()

        return np.random.choice(3, p=adjusted_probs)

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.11/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


AttributeError: 'MarketMAMLTrainer' object has no attribute '_init_normalization'

In [ ]:
def get_sleep_time(interval_minutes, market_start_hour=9, market_start_minute=15, market_close_hour=15, market_close_minute=0):
    # Get current time in IST
    now = datetime.now(pytz.utc).astimezone(ist_timezone)

    # Define the market start and close times in IST for today
    market_start_time = now.replace(hour=market_start_hour, minute=market_start_minute, second=0, microsecond=0)
    market_close_time = now.replace(hour=market_close_hour, minute=market_close_minute, second=0, microsecond=0)

    if now < market_start_time:
        # If current time is before market starts, set next_run_time to market start time
        next_run_time = market_start_time
    elif now > market_close_time:
        # If current time is after market close, calculate the time until the next market open
        next_market_start_time = market_start_time + timedelta(days=1)
        next_run_time = next_market_start_time
    else:
        # Calculate the minutes since the market start time
        minutes_since_market_start = (now - market_start_time).total_seconds() // 60
        # Calculate the number of minutes to the next interval boundary
        minutes_to_next_interval = interval_minutes - (minutes_since_market_start % interval_minutes)
        # Calculate the next run time by adding these minutes to the current time
        next_run_time = (now + timedelta(minutes=minutes_to_next_interval)).replace(second=0, microsecond=0)

    # Calculate the sleep time in seconds
    sleep_time = (next_run_time - now).total_seconds()
    return sleep_time

In [ ]:
def fetch_option_chain():
    while True:
        try:
            data = {
                "symbol": nf_index_symbol,
                "strikecount": 2,
                "timestamp": ""
            }
            response = fyers.optionchain(data=data)

            if response is not None:
                return response
        except Exception as e:
            print(f"Error fetching Option Chain: {e}")
            time.sleep(active_order_sleep)

index_oc= fetch_option_chain()

pd.DataFrame(index_oc['data']['optionsChain'])

In [ ]:
def assign_ce_pe_option_symbols():
    symbol_oc = fetch_option_chain()

    if symbol_oc != None:
        # Convert the response data into a DataFrame
        oc_df = pd.DataFrame(symbol_oc['data']['optionsChain'])

        # Find the first 'CE' symbol from the top
        first_ce_symbol = None
        for index, row in oc_df.iterrows():
            if row['option_type'] == 'CE':
                first_ce_symbol = row['symbol']
                first_ce_strike = row['strike_price']
                break

        # Find the first 'PE' symbol from the bottom
        first_pe_symbol = None
        for index, row in oc_df[::-1].iterrows():  # Iterate in reverse
            if row['option_type'] == 'PE':
                first_pe_symbol = row['symbol']
                first_pe_strike = row['strike_price']
                break

        return first_ce_symbol, first_pe_symbol, first_ce_strike, first_pe_strike

In [ ]:
def onmessage_ce(ce_message):
    global ce_ltp, index_ltp, unsubscribe_done
    try:
        if ce_message['symbol'] == ce_symbol:
            if "ltp" in ce_message:
                ce_ltp = ce_message["ltp"]
                ce_ltp = float(ce_ltp)

        elif ce_message['symbol'] == index_symbol:
            if "ltp" in ce_message:
                index_ltp = ce_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [ce_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {ce_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessageCE): {e}")


def onerror_ce(message):
    print("CE Error:", message)


def onclose_ce(message):
    print("CE Connection closed:", message)


def onopen_ce():

    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [ce_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def ce_buy_sell_ltp():
    global buy_sell_checked, ce_symbol, ce_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching CE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if ce_symbol is not None and ce_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                ce_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_ce,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_ce,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_ce,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_ce             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                ce_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def onmessage_pe(pe_message):
    global pe_ltp, index_ltp, unsubscribe_done
    try:
        if pe_message['symbol'] == pe_symbol:
            if "ltp" in pe_message:
                pe_ltp = pe_message["ltp"]
                pe_ltp = float(pe_ltp)

        elif pe_message['symbol'] == index_symbol:
            if "ltp" in pe_message:
                index_ltp = pe_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [pe_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {pe_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessagePE): {e}")

def onerror_pe(message):
    print("PE Error:", message)


def onclose_pe(message):
    print("PE Connection closed:", message)


def onopen_pe():
    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [pe_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def pe_buy_sell_ltp():
    global buy_sell_checked, pe_symbol, pe_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching PE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if pe_symbol is not None and pe_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                pe_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_pe,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_pe,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_pe,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_pe             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                pe_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def place_order(symbol):
    try:
        market_order_data = {
            "symbol": symbol,
            "qty": int(quantity),
            "type": 2,  # Market Order
            "side": 1,
            "productType": "INTRADAY",
            "limitPrice": 0,
            "stopPrice": 0,
            "validity": "DAY",
            "disclosedQty": 0,
            "offlineOrder":False
        }

        market_order_entry = fyers.place_order(data=market_order_data)

        if "id" in market_order_entry:
            market_order_id = market_order_entry["id"]
            market_order_message = market_order_entry["message"]
            print(f"{market_order_message}")

    except Exception as e:
        print(f"Error placing orders: {str(e)}")

In [ ]:
def trail_order(symbol, stoploss):
    while True:
        try:
            stoploss = int(stoploss)
            pending_order = fyers.orderbook()

            matching_orders = [order for order in pending_order["orderBook"] if order["status"] == 6]

            modified_orders = 0

            for order in matching_orders:
                if order['symbol'] == symbol:
                    pending_order_id = order['id']
                    pending_order_side = order['side']
                    pending_order_side = int(pending_order_side)

                    if pending_order_side != 1:
                        data = {
                            "id": pending_order_id,
                            "type": 4,
                            "limitPrice": stoploss - 1,
                            "stopPrice": stoploss
                        }

                        modify = fyers.modify_order(data=data)
                        trail_message = modify["message"]
                        print(f"{trail_message}")

                        if trail_message == "Successfully modified order":
                            modified_orders += 1

            # Check if all matching orders are successfully modified
            if modified_orders == len(matching_orders):
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print("Error modifying order:" + str(e))

In [ ]:
def exit_active_order(symbol):
    while True:
        try:
            data = {
                "id":f"{symbol}-INTRADAY"
            }

            exit_response = fyers.exit_positions(data=data)

            if ["message"] in exit_response:
                print(exit_response["message"])
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print(f"Error exiting Order: {e}")

In [ ]:
def reset_flags():
    global active_order, buy_sell_checked

    active_order = False
    buy_sell_checked = False

In [ ]:
# Function to save profits and losses
def save_overall(overall_win, overall_loss, capital):
    trade_type = {
        "overall_win": overall_win,
        "overall_loss": overall_loss,
        "capital": capital
    }

    with open("trade_data.json", "w") as file:
        json.dump(trade_type, file)


# Function to load wins and losses
def load_overall():
    try:
        with open('trade_data.json') as file:
            return json.load(file)
    except FileNotFoundError:
        return None

In [ ]:
def handle_active_ce_order():
    def ce_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, ce_ltp, index_ltp, ce_strike, ce_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        ce_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if ce_ltp != 0 and index_ltp != 0:
                    ce_ltp_array.append(ce_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)


                    if index_ltp <= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(ce_symbol)

                        points = int(ce_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("CE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp >= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                            trailing_sl_inside = int(ce_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp - stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                    else:
                        if (index_ltp - prev_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(ce_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp - trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("CE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=ce_order_loop()).start()

In [ ]:
def handle_active_pe_order():
    def pe_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, pe_ltp, index_ltp, pe_strike, pe_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        pe_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if pe_ltp != 0 and index_ltp != 0:
                    pe_ltp_array.append(pe_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)

                    if index_ltp >= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(pe_symbol)

                        points = int(pe_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("PE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp_array}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp <= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                            trailing_sl_inside = int(pe_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp + stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                    else:
                        if (prev_ltp - index_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(pe_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp + trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("PE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=pe_order_loop()).start()

In [ ]:
def ce_entry():
    threading.Thread(target=ce_buy_sell_ltp).start()

    def ce_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if ce_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = ce_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp + target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp - trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(ce_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_ce_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=ce_entry_thread()).start()

In [ ]:
def pe_entry():
    threading.Thread(target=pe_buy_sell_ltp).start()

    def pe_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if pe_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = pe_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp - target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp + trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(pe_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_pe_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=pe_entry_thread()).start()

In [ ]:
def market_entry_exit_logic(action, actual_closing_price, final_df):
    global sl_hit_condition, unsubscribe_done, ce_ltp, pe_ltp, index_ltp, fixed_ltp, fixed_index_ltp, prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, ce_strike, pe_strike, ce_symbol, pe_symbol

    ce_ltp = 0
    pe_ltp  =0
    index_ltp = 0
    fixed_ltp = 0
    fixed_index_ltp = 0
    prev_ltp = 0
    target_inside = 0
    target_index_inside = 0
    trailing_sl_inside = 0
    trailing_index_inside = 0
    ce_strike = None
    pe_strike = None
    ce_symbol = None
    pe_symbol = None

    final_current_time = final_df.index[-1].time()
    print(final_current_time)

    # Ensure trading occurs only during market hours
    if final_current_time >= dt_time(9, (15 + interval_minutes)) and final_current_time <= dt_time(18, 0):
        if action == 1 and not active_order:  # Buy action
            print("Buy action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making CE Position at {actual_closing_price}")
            ce_entry()  # Execute CE entry logic
        elif action == 2 and not active_order:  # Sell action
            print("Sell action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making PE Position at {actual_closing_price}")
            pe_entry()  # Execute PE entry logic
        elif action == 0:
            print("Hold action detected by PPO model. No trade executed.")

In [ ]:
def fetch_and_prepare_final_data():
    final_df = fetch_candle_data(10)

    final_data_processor = DataProcessor(final_df, live_processing=True)

    final_processed_df = final_data_processor.process().df

    return final_processed_df

In [ ]:
def find_local_extrema(df):
    order=5
    atr_multiplier=1.5
    min_distance=5

    # Find local maxima and minima
    local_max = argrelextrema(df['high'].values, np.greater_equal, order=order)[0]
    local_min = argrelextrema(df['low'].values, np.less_equal, order=order)[0]

    # Calculate the threshold based on ATR
    threshold = df['atr'] * atr_multiplier

    # Filter by significance
    significant_max = []
    significant_min = []

    for idx in local_max:
        if idx > order and idx < len(df) - order:
            high = df['high'].iloc[idx]
            if significant_min:
                low = df['low'].iloc[significant_min[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_max.append(idx)
            else:
                significant_max.append(idx)

    for idx in local_min:
        if idx > order and idx < len(df) - order:
            low = df['low'].iloc[idx]
            if significant_max:
                high = df['high'].iloc[significant_max[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_min.append(idx)
            else:
                significant_min.append(idx)

    # Ensure minimum distance
    def filter_by_distance(points, min_distance):
        filtered_points = []
        for i in range(len(points)):
            if not filtered_points or (points[i] - filtered_points[-1]) > min_distance:
                filtered_points.append(points[i])
        return filtered_points

    significant_max = filter_by_distance(significant_max, min_distance)
    significant_min = filter_by_distance(significant_min, min_distance)

    return significant_max, significant_min

In [ ]:
def get_trendline(df, point1, point2, kind='high'):
    x = [point1, point2]
    if kind == 'high':
        y = df['high'].values[x]
    else:
        y = df['low'].values[x]
    coeffs = np.polyfit(x, y, 1)
    trendline = np.polyval(coeffs, range(len(df)))
    return trendline

In [ ]:
# Online training configuration
online_learning_steps = 1000  # Steps for each retraining
retrain_interval_minutes = 60  # Retrain the model every 60 minutes

start_time = datetime.now()

while True:
    clear_output(wait=True)

    num_candles = 100

    final_df = fetch_and_prepare_final_data()

    final_df = final_df.iloc[:-1]

    target = final_df['Target'].iloc[-1]
    trailing_sl = final_df['Stop Loss'].iloc[-1]

    # Initialize the TradingEnvironment with the latest data
    live_env = create_env(final_df, config)
    obs = live_env.reset()

    # Use PPO model to predict action for the latest data point
    action, _ = ppo_model.predict(obs, deterministic=True)

    # Extract the actual closing price
    actual_closing_price = final_df['close'].iloc[-1]

    # Retrain PPO model at specified intervals
    if (datetime.now() - start_time).seconds >= retrain_interval_minutes * 60:
        print("Retraining PPO model with online data...")
        train_env = TradingEnvironment(final_df, config)  # Create a new training environment
        ppo_model.set_env(train_env)  # Update the PPO model's environment
        ppo_model.learn(total_timesteps=online_learning_steps)  # Retrain the model
        ppo_model.save(model_save_path)  # Save the updated model
        start_time = datetime.now()

    # Identify most recent high and low points
    recent_highs, recent_lows = find_local_extrema(final_df)

    most_recent_high = recent_highs[-1] if len(recent_highs) > 1 else None
    most_recent_low = recent_lows[-1] if len(recent_lows) > 1 else None

    high_trendline = [np.nan] * len(final_df)
    low_trendline = [np.nan] * len(final_df)

    if most_recent_high is not None:
        previous_high = recent_highs[-2] if len(recent_highs) > 2 else most_recent_high
        high_trendline = get_trendline(final_df, previous_high, most_recent_high, kind='high')

    if most_recent_low is not None:
        previous_low = recent_lows[-2] if len(recent_lows) > 2 else most_recent_low
        low_trendline = get_trendline(final_df, previous_low, most_recent_low, kind='low')

    # Prepare candlestick data for mplfinance
    actual_candles = final_df[-num_candles:].copy()

    # Create a DataFrame for mplfinance
    mpf_df = actual_candles[['open', 'high', 'low', 'close']]

    # Create addplot elements for predicted prices and actual close prices
    ap = [
        mpf.make_addplot(final_df['close'][-num_candles:], color='none', panel=0, secondary_y=False, label=f"Actual Price: {final_df['close'].iloc[-1]}"),
        #mpf.make_addplot(y_pred_ensemble_final_plot, color=(0.95, 0.38, 0.25, 1), panel=0, secondary_y=False, label=f'Predicted Prices ({y_pred_ensemble_final_plot[-1]:.2f})')
    ]

    # Add trendlines to the plot
    if most_recent_high is not None:
        ap.append(mpf.make_addplot(high_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    if most_recent_low is not None:
        ap.append(mpf.make_addplot(low_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    fig, axlist = mpf.plot(mpf_df, type='candle', style='binancedark', volume=False, addplot=ap,
                        title=f'Chart', ylabel='Price',
                        figsize=(14, 7), returnfig=True)

    for ax in axlist:
        ax.grid(False)

    axlist[0].legend()
    plt.show()

    # Execute market entry/exit logic
    market_entry_exit_logic(action, actual_closing_price, final_df)

    sleep_time = get_sleep_time(interval_minutes)
    print(f"Sleeping for {sleep_time}")
    time.sleep(sleep_time)